In [3]:
#https://github.com/sgl-project/sglang/issues/391
#python -m sglang.launch_server --model-path dillonlaird/hf-llava-v1.6-34b --port 30000
#https://huggingface.co/dillonlaird/hf-llava-v1.6-34b
#!pip install llama_index

In [5]:
from pathlib import Path
from llama_index.core import SimpleDirectoryReader
import os
import numpy as np
from tqdm import tqdm
import requests
from transformers import AutoTokenizer
import sglang as sgl

In [6]:
# load as image documents
image_documents = SimpleDirectoryReader("/home/data/ISIC2019", required_exts=[".jpeg", ".jpg", ".png"], recursive=True, filename_as_id=True).load_data()

In [7]:
def metadata_str2dict(string):
    # Split the string into lines
    lines = string.strip().split("\n")
    
    # Create a dictionary to store the key-value pairs
    file_info_dict = {}
    
    # Parse each line and add key-value pairs to the dictionary
    for line in lines:
        key, value = line.split(": ", 1)
        file_info_dict[key] = value
    
    return(file_info_dict)

In [8]:
image_patch = []
image_no_patch = []

for image in image_documents:
    metadata = metadata_str2dict(image.get_metadata_str())
    image_path = metadata['file_path']

    folder_name = os.path.basename(os.path.dirname(image_path))
    
    if folder_name == "patch":
        metadata['patch'] = True
        image_patch.append({'image': image, 'meta': metadata})


    elif folder_name == "no_patch":
        metadata['patch'] = False
        image_no_patch.append({'image': image, 'meta': metadata})


    else:
        print("The jpg is not in either the patch or no_patch folder.")

In [9]:
print(image_patch[0]['meta'])

{'file_path': '/home/data/ISIC2019/patch/ISIC_0000557.jpg', 'file_name': 'ISIC_0000557.jpg', 'file_type': 'image/jpeg', 'file_size': '2630625', 'creation_date': '2024-05-12', 'last_modified_date': '2024-05-12', 'patch': True}


In [10]:
def generate(image_path: str, prompt: str, tokenizer):
    chat = [
        {"role": "system", "content": "Answer the question."},
        {"role": "user", "content": "<image>\n" + prompt},
    ]
    chat_str = tokenizer.apply_chat_template(chat, tokenize=False)
    chat_str += "<|img_start|>assistant\n"
    sampling_params = {"temperature": 0.2, "max_new_tokens": 1536}
    res = requests.post(
        "http://localhost:30000/generate",
        json={
            "text": chat_str,
            "image_data": image_path,
            "sampling_params": sampling_params,
        },
    )
    return res.json()["text"]

In [12]:
@sgl.function
def image_qa(s, image_file, question):
    s += sgl.user(sgl.image(image_file) + question)
    s += sgl.assistant(sgl.gen("answer", max_tokens=256))

In [13]:
num_imgs = 100
num_imgs_per_class = int(np.ceil(num_imgs/2))

# combine images with and without patch
images = []
images.extend(image_patch[:num_imgs_per_class])
images.extend(image_no_patch[:num_imgs_per_class])

In [14]:
#tokenizer = AutoTokenizer.from_pretrained("liuhaotian/llava-v1.6-34b")
prompts = [#"The image shows a part of skin with a lesion. Often a patch marks if it is cancer. I want to figure out if this image has such a patch. The patch is usually colourful and has an ellipsoid shape. Do you see a Patch? Please answer with one word, yes or no.",
          #"The image shows a part of skin with a lesion. Sometimes a patch marks if it is cancer. I want to figure out if this image has such a patch. The patch is usually colourful and has an ellipsoid shape. Do you see a Patch? Please answer with one word, yes or no.",
          #"The image shows a lesion and sometimes a coloful ellipsoid patch is placed next to it. Does this image has a patch? Answer with yes or no, with one word.",
          #"You see an image of a part of a skin with a lesion. To mark cancer often a patch is attached next to the lesion. This patch is colorful and ellipsoid. It is not part of the skin. Not evry picture has such a patch. I want to figure out if this image shows a patch next to the lesion. Do you see a patch? Answer with one word of [yes, no].",
          #"You see an image of a part of a skin with a lesion. To mark cancer sometimes a patch is attached next to the lesion. This patch is colorful and ellipsoid. It is not part of the skin. Not evry picture has such a patch. I want to figure out if this image shows a patch next to the lesion. Do you see a patch? Answer with one word of [yes, no].",
          #"I show you images of a part of a skin with a lesion. I want to figure out if a colorful, ellipsoid patch is attached next to the lesion to mark the lesion as malignant. Do you see a colorful, ellipsoid patch attached on the skin? Answer with one of the options [yes, no]", 
          #"I show you images of a part of a skin with a lesion. I want to figure out if a colorful, ellipsoid patch is attached next to the lesion to mark the lesion as malignant. Do you see a colorful, ellipsoid patch attached on the skin? Answer with one word of the options [yes, no]"
          #"The image shows a lesion and sometimes a coloful ellipsoid patch is placed next to it. The patch does not belong to the skin. Does this image has a patch? Answer with yes or no, with one word.",
          #"The image shows part of a skin with a lesion. Sometimes a coloful ellipsoid patch is placed next to the lesion to mark if it is malignant. The patch does not belong to the skin. Do you see a colorful, ellipsoid patch attached on the skin. Answer with one word of the options [yes, no].",
          "The image shows part of a skin with a lesion. Sometimes a coloful ellipsoid patch is placed next to the lesion to mark if it is malignant. The patch does not belong to the skin. Do you see a colorful, ellipsoid patch attached on the skin or not? Answer with one word of the options [yes, no].", 
          #"The image shows a part of skin with a lesion. Often a patch marks if it is cancer. I want to figure out if this image has such a patch. Do you see a patch? Please answer with yes or no. Please answer with one word.",
          #"Is a patch in the image? Please answer with one word, yes or no.",
          #"Is a patch in the image? Answer with yes or no.",
          #"The image shows a part of skin with a lesion. Often a patch marks if it is cancer. I want to figure out if this image has such a patch. Do you see a patch? Please answer with one word, yes or no.",
          #"Can you see a patch?",
          #"Can you see a patch? Reply with one word, yes or no.",
          #"patch?",
          #"patch? One word reply."
         ]

for prompt in prompts:
    correct_classified = 0
    false_classified = 0
    more_than_one_word = 0
    num_yes_patch = 0
    num_no_no_patch = 0
    num_yes = 0
    num_no = 0

    print("-----------------------------------------\n")
    print(prompt)
    
    for image_dict in tqdm(images[:num_imgs]):
        image = image_dict['image']
        meta = image_dict['meta']
        
        ## display image
        #imageUrl = meta['file_path']
        #image = Image.open(imageUrl).convert("RGB")
        #plt.figure(figsize=(8, 3))
        #plt.imshow(image)
    
        # prompt 
        prompt= prompt
    
        # image
        image_path = meta['file_path']

        #response
        #desc = generate(image_path, prompt, tokenizer)
        desc = image_qa('', image_path, prompt)

        print(desc)

    
        response = str(desc).lower().replace(" ", "")
        if response == "yes":
            #print("yes")
            num_yes += 1
            if meta['patch']:
                correct_classified += 1
                num_yes_patch += 1
                #print("correct")
            else:
                false_classified += 1
                #print("false")
    
        elif response == "no":
            #print("no")
            
            num_no += 1
            if meta['patch']:
                false_classified += 1
                #print("false")
            else:
                correct_classified += 1
                num_no_no_patch +=1
                #print("correct")
        else:
            #print("No yes or no!")
            more_than_one_word += 1

    acc = correct_classified/(num_imgs - more_than_one_word)
    
    print("\nAccuracy: ",acc)
    print("correct classified: ", correct_classified,"\nfalse_classified: ", false_classified, 
          "\nNumber yes: ", num_yes, "\nNumber no: ", num_no, "\nNeither yes or no: ", more_than_one_word,
          "\nNumber True positives: ", num_yes_patch, "\nNumber True negatives: ", num_no_no_patch,
          "\nnum images: ", num_imgs)
    print("\n-----------------------------------------\n\n")

/usr/local/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


-----------------------------------------

The image shows part of a skin with a lesion. Sometimes a coloful ellipsoid patch is placed next to the lesion to mark if it is malignant. The patch does not belong to the skin. Do you see a colorful, ellipsoid patch attached on the skin or not? Answer with one word of the options [yes, no].


  0%|                                                                                                                                      | 0/100 [00:00<?, ?it/s]


AssertionError: Please specify a backend